In [12]:
import requests
import time
import pandas as pd
from datetime import datetime

In [ ]:
# Configuration
API_ROOT = "https://ensembledata.com/apis"
ENDPOINT = "/reddit/subreddit/posts"
TOKEN = "YOUR_ACTUAL_TOKEN_HERE"  # Replace with your actual token

In [19]:
# Fields mapping - English column names for analysis
FIELD_MAPPING = {
    # Original Reddit field: (French description, English column name)
    'id': ('Post ID unique', 'id'),
    'subreddit': ('r/BDS ou r/BoycottIsrael', 'subreddit'),
    'permalink': ('URL pour vérification', 'permalink'),
    'title': ('Titre du post', 'title'),
    'selftext': ('Texte du post', 'selftext'),
    'created_utc': ('Timestamp UTC', 'created_utc'),
    'author': ('Auteur du post', 'author'),
    'score': ('Score total (upvotes - downvotes)', 'score'),
    'ups': ('Upvotes', 'ups'),
    'downs': ('Downvotes', 'downs'),
    'num_comments': ('Nombre de commentaires', 'num_comments'),
    'upvote_ratio': ('Ratio upvotes/total', 'upvote_ratio'),
    'author_fullname': ('ID Reddit de l\'auteur', 'author_fullname'),
    'subreddit_subscribers': ('Taille de la communauté', 'subreddit_subscribers'),
    'total_awards_received': ('Indicateur d\'impact', 'total_awards_received'),
    'link_flair_text': ('Catégorie/thème du post', 'link_flair_text'),
    'over_18': ('Contenu NSFW', 'over_18'),
    'locked': ('Discussion fermée', 'locked'),
    'created': ('Timestamp local', 'created_local'),
    'edited': ('Si édité', 'edited'),
}

# Pagination loop
params = {
    "name": "BoycottIsrael", #or BoycottIsrael
    "sort": "hot",
    "period": "all",
    "cursor": "",
    "token": TOKEN
}

request_count = 0
max_requests = 7  # 48 credits / 2 per request

# Initialize list to store all posts
all_posts_data = []

while request_count < max_requests:
    try:
        # Make request
        res = requests.get(API_ROOT + ENDPOINT, params=params)
        data = res.json()
        
        # Process posts
        posts = data['data']['posts']
        print(f"📄 Batch {request_count + 1} - {len(posts)} posts")
        
        # Extract data for each post
        for post in posts:
            post_data = post['data']
            row = {}
            
            # Extract all fields from mapping
            for field, (french_desc, english_col) in FIELD_MAPPING.items():
                value = post_data.get(field)
                
                # Special handling for certain fields
                if field == 'permalink' and value:
                    value = f"https://reddit.com{value}"
                elif field in ['created', 'created_utc'] and value:
                    # Add formatted date as separate column
                    formatted_col = f"{english_col}_formatted"
                    row[formatted_col] = datetime.fromtimestamp(value).strftime('%Y-%m-%d %H:%M:%S')
                
                row[english_col] = value
            
            # Add calculated fields
            if post_data.get('selftext'):
                row['selftext_length'] = len(post_data['selftext'])
            
            # Calculate downvotes if upvote_ratio is available
            if 'ups' in row and 'upvote_ratio' in row and row['upvote_ratio']:
                ups = row['ups']
                upvote_ratio = row['upvote_ratio']
                if upvote_ratio > 0:
                    # Calculate total votes and downvotes
                    # upvote_ratio = ups / (ups + downs)
                    total_votes = ups / upvote_ratio
                    row['downs_calculated'] = max(0, int(total_votes - ups))
            
            all_posts_data.append(row)
        
        # Check for next page
        next_cursor = data['data'].get('nextCursor')
        if not next_cursor:
            print("🎯 Fin de la pagination")
            break
        
        # Update cursor
        params['cursor'] = next_cursor
        request_count += 1
        
        # Small delay between requests
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        break

# Create DataFrame with English column names
df = pd.DataFrame(all_posts_data)

# Display DataFrame info
print(f"\n✅ Terminé. {request_count} requêtes effectuées.")
print(f"📊 DataFrame créé avec {len(df)} posts")
print("\n" + "="*50)
print("DATAFRAME INFO:")
print("="*50)
print(f"Shape: {df.shape} (rows, columns)")
print(f"Columns: {len(df.columns)}")

📄 Batch 1 - 25 posts
📄 Batch 2 - 25 posts
📄 Batch 3 - 25 posts
📄 Batch 4 - 25 posts
📄 Batch 5 - 25 posts
❌ Erreur: 'data'

✅ Terminé. 5 requêtes effectuées.
📊 DataFrame créé avec 125 posts

DATAFRAME INFO:
Shape: (125, 24) (rows, columns)
Columns: 24


In [20]:
# Show first few rows
print("\nFIRST 3 ROWS:")
print("="*80)
print(df[['id', 'title', 'author', 'ups', 'downs', 'score', 'num_comments', 'link_flair_text']].head(3))
print("="*80)


FIRST 3 ROWS:
        id                                              title  \
0  1nqfbjv  A lot of people asking about tech and gaming c...   
1  1qh8z2s                   Dr Jartt colour correcting dupe?   
2  1qgwsvx  Should I go to a concert at the walt Disney co...   

                 author  ups  downs  score  num_comments  \
0           ibrahim_D12   62      0     62            32   
1     That-Flamingo9866    8      0      8             3   
2  ConfidenceCareless71   17      0     17             6   

           link_flair_text  
0                     None  
1  Alternative Suggestions  
2                 Question  


In [21]:
# Show basic statistics
print("\n📈 BASIC STATISTICS:")
print("="*50)
if 'ups' in df.columns:
    print(f"Average upvotes: {df['ups'].mean():.0f}")
if 'downs_calculated' in df.columns:
    print(f"Average downvotes (calculated): {df['downs_calculated'].mean():.0f}")
if 'score' in df.columns:
    print(f"Average score: {df['score'].mean():.0f}")
print(f"Average comments: {df['num_comments'].mean():.1f}")
print(f"Unique authors: {df['author'].nunique()}")
print(f"Unique posts: {df['id'].nunique()}")


📈 BASIC STATISTICS:
Average upvotes: 68
Average downvotes (calculated): 1
Average score: 68
Average comments: 6.8
Unique authors: 92
Unique posts: 125


In [22]:
# Save to CSV with English column names
csv_filename = "boycottisrael_posts_hot.csv" #or BoycottIsrael
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\n💾 Data saved to: {csv_filename}")


💾 Data saved to: boycottisrael_posts_hot.csv


In [23]:
# Create a reference file with French descriptions
ref_filename = "column_descriptions_boycottisrael_hot.txt"
with open(ref_filename, 'w', encoding='utf-8') as f:
    f.write("COLUMN DESCRIPTIONS - French to English\n")
    f.write("="*60 + "\n")
    for field, (french_desc, english_col) in FIELD_MAPPING.items():
        f.write(f"{english_col:25} = {french_desc}\n")
    f.write("\n" + "="*60 + "\n")
    f.write(f"Total rows: {len(df)}\n")
    f.write(f"Total columns: {len(df.columns)}\n")

print(f"📝 Column descriptions saved to: {ref_filename}")

📝 Column descriptions saved to: column_descriptions_boycottisrael_hot.txt
